In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_validate, learning_curve, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, matthews_corrcoef,
    classification_report, confusion_matrix, roc_curve
)

from src.preprocessing import load_data, preprocess, split_and_scale
from src.model import build_baseline, tune_hyperparameters, PARAM_GRID
from src.evaluation import (
    compute_metrics, plot_confusion_matrix, plot_roc_curve,
    plot_feature_importance, plot_learning_curve
)

os.makedirs('plots', exist_ok=True)

RANDOM_STATE = 42
DATA_PATH = 'data/heart.csv'

ModuleNotFoundError: No module named 'numpy'

## 1. Dane

303 pacjentów, 13 cech, target binarny (0 = brak choroby, 1 = choroba). Kilka brakujących wartości w `ca` i `thal` ukrytych jako `?`.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Wymiary: {df_raw.shape}')
df_raw.head()

In [ ]:
# typy, braki, zakres
info = pd.DataFrame({
    'dtype': df_raw.dtypes,
    'non_null': df_raw.count(),
    'null': df_raw.isnull().sum(),
    'unique': df_raw.nunique()
})
# '?' liczymy osobno bo CSV może je wczytać jako string
for col in df_raw.select_dtypes('object').columns:
    n_question = (df_raw[col] == '?').sum()
    if n_question:
        print(f"  '{col}': {n_question} wartości '?'")
info

In [ ]:
df_raw.describe()

## 2. EDA

Szybki przegląd rozkładu klas, korelacji i rozkładów cech numerycznych.

In [ ]:
# rozkład targetu
target_counts = df_raw['target'].replace({'?': np.nan}).astype(float)
# target w raw może być 0-4, binaryzujemy
target_binary = (target_counts > 0).astype(int)
vc = target_binary.value_counts()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].pie(vc, labels=['Brak choroby', 'Choroba'], autopct='%1.1f%%', colors=['#2E5FA3', '#E8834A'])
axes[0].set_title('Rozkład klas')
vc.plot(kind='bar', ax=axes[1], color=['#2E5FA3', '#E8834A'], edgecolor='white')
axes[1].set_xticklabels(['Brak choroby', 'Choroba'], rotation=0)
axes[1].set_title('Liczba próbek')
axes[1].set_ylabel('N')
plt.tight_layout()
plt.show()
print(f'Brak choroby: {vc[0]}  |  Choroba: {vc[1]}  |  Balans: {vc[1]/vc.sum():.1%}')

In [ ]:
# korelacja - żeby wiedzieć które cechy są interesujące
# najpierw czyścimy dane do liczb (jak robi preprocessing.py)
df_clean = df_raw.copy()
for col in df_clean.columns:
    df_clean[col] = pd.to_numeric(df_clean[col].replace('?', np.nan), errors='coerce')
df_clean['target'] = (df_clean['target'] > 0).astype(int)
df_clean = df_clean.fillna(df_clean.median(numeric_only=True))

corr = df_clean.corr()
plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Macierz korelacji Pearsona')
plt.tight_layout()
plt.show()

In [ ]:
# cechy numeryczne - histogramy z podziałem na klasę
numeric_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, feat in zip(axes, numeric_features):
    for label, color in [(0, '#2E5FA3'), (1, '#E8834A')]:
        subset = df_clean[df_clean['target'] == label][feat]
        ax.hist(subset, bins=20, alpha=0.6, color=color,
                label=['Brak', 'Choroba'][label])
    ax.set_title(feat)
    ax.legend(fontsize=8)
plt.suptitle('Rozkłady cech numerycznych (per klasa)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# cechy kategoryczne vs target
cat_features = ['cp', 'sex', 'fbs', 'restecg', 'exang', 'slope']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.flat, cat_features):
    ct = pd.crosstab(df_clean[feat], df_clean['target'], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=['#2E5FA3', '#E8834A'], edgecolor='white', legend=False)
    ax.set_title(feat)
    ax.set_ylabel('% w klasie')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)
axes[0, 0].legend(['Brak choroby', 'Choroba'], fontsize=9)
plt.suptitle('Cechy kategoryczne — % choroba per wartość', y=1.02)
plt.tight_layout()
plt.show()

## 3. Preprocessing

Czyszczenie `?` -> NaN -> mediana, binaryzacja targetu, split 80/20 stratified, StandardScaler.

In [ ]:
df = load_data(DATA_PATH)
X, y = preprocess(df)
X_train, X_test, X_train_s, X_test_s, y_train, y_test, scaler = split_and_scale(X, y)

print(f'Train: {X_train.shape[0]} próbek  |  Test: {X_test.shape[0]} próbek')
print(f'Cechy: {X_train.shape[1]}')
print(f'Target train: {y_train.value_counts().to_dict()}')
print(f'Target test:  {y_test.value_counts().to_dict()}')

## 4. Model bazowy

Random Forest, 100 drzew, domyślne parametry. Punkt odniesienia przed tuningiem.

In [ ]:
baseline = build_baseline()
baseline.fit(X_train_s, y_train)

metrics_baseline = compute_metrics(baseline, X_test_s, y_test)
print('=== Baseline ===' )
for k, v in metrics_baseline.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')

## 5. Tuning hiperparametrów

GridSearchCV, 5-fold CV, scoring=f1. Grid: n_estimators × max_depth × min_samples_split × min_samples_leaf × max_features = 72 kombinacje × 5 foldów = 360 fitów. Trwa ~2 minuty.

In [ ]:
print(f'Grid size: {1}')
import functools, operator
grid_size = functools.reduce(operator.mul, [len(v) for v in PARAM_GRID.values()], 1)
print(f'Kombinacji: {grid_size}  |  Fitów: {grid_size * 5}')

_, best_model = tune_hyperparameters(X_train_s, y_train)

print('\nNajlepsze parametry:')
for k, v in best_model.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nCV F1 (best): {best_model.best_score_:.4f}')

## 6. Ewaluacja modelu po tuningu

In [ ]:
from src.model import cross_validate as cv_model

tuned = best_model.best_estimator_
metrics_tuned = compute_metrics(tuned, X_test_s, y_test)

print('=== Model po tuningu (zbiór testowy) ===')
for k, v in metrics_tuned.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')

print('\n=== Classification Report ===')
y_pred = tuned.predict(X_test_s)
print(classification_report(y_test, y_pred, target_names=['Brak choroby', 'Choroba']))

In [ ]:
cv_results = cv_model(tuned, X_train_s, y_train)
print('=== 5-fold Cross-Validation (zbiór treningowy) ===')
for metric, scores in cv_results.items():
    print(f'  {metric}: {scores.mean():.4f} ± {scores.std():.4f}')

## 7. Wizualizacje

In [ ]:
# macierz pomyłek
plot_confusion_matrix(tuned, X_test_s, y_test, save_path='plots/confusion_matrix.png')
plt.show()

In [ ]:
# krzywa ROC
plot_roc_curve(tuned, X_test_s, y_test, save_path='plots/roc_curve.png')
plt.show()

In [ ]:
# ważność cech
feature_names = X.columns.tolist()
plot_feature_importance(tuned, feature_names, save_path='plots/feature_importance.png')
plt.show()

In [ ]:
# krzywa uczenia - czy mamy over/underfitting
plot_learning_curve(tuned, X_train_s, y_train, save_path='plots/learning_curve.png')
plt.show()

## 8. Porównanie: baseline vs po tuningu

In [ ]:
metric_names = ['accuracy', 'f1', 'roc_auc', 'mcc']
labels_pl = ['Accuracy', 'F1-Score', 'ROC-AUC', 'MCC']

vals_baseline = [metrics_baseline[m] for m in metric_names]
vals_tuned    = [metrics_tuned[m]    for m in metric_names]

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, vals_baseline, width, label='Baseline', color='#2E5FA3', alpha=0.85)
bars2 = ax.bar(x + width/2, vals_tuned,    width, label='Po tuningu', color='#E8834A', alpha=0.85)

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels_pl)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Wartość metryki')
ax.set_title('Baseline vs Model po tuningu (zbiór testowy)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# tabela podsumowania
summary = pd.DataFrame({
    'Metryka': labels_pl,
    'Baseline': [f'{v:.4f}' for v in vals_baseline],
    'Po tuningu': [f'{v:.4f}' for v in vals_tuned],
    'Delta': [f'{(t-b):+.4f}' for b, t in zip(vals_baseline, vals_tuned)]
})
summary